# LangChain Tool Calling Workflow

This notebook demonstrates how to properly define tools, bind them to an LLM, dynamically execute tool calls, and pass tool results back for a final response.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_groq import ChatGroq

load_dotenv()

# 1. Define tool with decorator and docstring
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    return f"The current weather in {location} is 72°F and sunny."

tools = [get_weather]
tools_by_name = {t.name: t for t in tools}

# 2. Initialize Model with bound tools
model = ChatGroq(model="llama-3.3-70b-versatile")
model_with_tools = model.bind_tools(tools)

In [ ]:
# Step 1: Model generates tool calls
messages = [HumanMessage(content="What's the weather in Boston?")]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools dynamically and collect results
for tool_call in ai_msg.tool_calls:
    # Match tool by name and invoke passing the full tool_call dict
    selected_tool = tools_by_name[tool_call["name"]]
    tool_result = selected_tool.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)

# Use .content to retrieve textual response
print(final_response.content)